In [1]:
import os

data_dir = './data'
print(f"Folder exists: {os.path.exists(data_dir)}")
print(f"Absolute path: {os.path.abspath(data_dir)}")
print(f"Full path: {os.path.join(os.getcwd(), 'data')}")

# List what's inside (if it exists)
if os.path.exists(data_dir):
    files = os.listdir(data_dir)
    print(f"Files inside: {files}")


Folder exists: True
Absolute path: C:\Users\kanks\OneDrive\Desktop\4see Demo AI\data
Full path: C:\Users\kanks\OneDrive\Desktop\4see Demo AI\data
Files inside: ['archive']


In [2]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("STUDENT DROPOUT PREDICTION - DATA LOADING")
print("="*70 + "\n")

# ========== CHECK FOLDER STRUCTURE ==========
data_dir = './data'

print(f"📁 Current working directory: {os.getcwd()}")
print(f"📁 Data folder: {os.path.abspath(data_dir)}\n")

# List all files
print("Files in ./data/:")
for root, dirs, files in os.walk(data_dir):
    level = root.replace(data_dir, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    
    for file in files:
        if file.endswith('.csv'):
            filepath = os.path.join(root, file)
            size = os.path.getsize(filepath) / 1024  # KB
            print(f'{indent}  ├─ {file} ({size:.1f} KB)')

print("\n" + "="*70)
print("LOADING DATASETS")
print("="*70 + "\n")

# ========== LOAD DATASETS ==========
datasets = {}

# Dataset 1: Student Math
try:
    path = os.path.join(data_dir, 'archive', 'student-mat.csv')
    if os.path.exists(path):
        df1 = pd.read_csv(path, sep=';')
        datasets['student_math'] = df1
        print(f"✅ Student Math: {df1.shape[0]} rows × {df1.shape[1]} columns")
    else:
        print(f"⚠️ student-mat.csv not found at {path}")
except Exception as e:
    print(f"❌ Error loading student-mat.csv: {e}")

# Dataset 2: Student Portuguese
try:
    path = os.path.join(data_dir, 'archive', 'student-por.csv')
    if os.path.exists(path):
        df2 = pd.read_csv(path, sep=';')
        datasets['student_portuguese'] = df2
        print(f"✅ Student Portuguese: {df2.shape[0]} rows × {df2.shape[1]} columns")
    else:
        print(f"⚠️ student-por.csv not found")
except Exception as e:
    print(f"❌ Error loading student-por.csv: {e}")

# Dataset 3: Student Merge
try:
    path = os.path.join(data_dir, 'archive', 'student-merge.csv')
    if os.path.exists(path):
        df3 = pd.read_csv(path)
        datasets['student_merge'] = df3
        print(f"✅ Student Merge: {df3.shape[0]} rows × {df3.shape[1]} columns")
    else:
        print(f"⚠️ student-merge.csv not found")
except Exception as e:
    print(f"❌ Error loading student-merge.csv: {e}")

# ========== EXPLORE DATA ==========
print("\n" + "="*70)
print("DATA EXPLORATION")
print("="*70 + "\n")

for name, df in datasets.items():
    print(f"📊 {name.upper()}")
    print(f"   Shape: {df.shape}")
    print(f"   Columns: {list(df.columns)}")
    print(f"   Missing values: {df.isnull().sum().sum()}")
    
    # Check for target column
    target_cols = [col for col in df.columns if 'target' in col.lower() or 'dropout' in col.lower()]
    if target_cols:
        print(f"   Target columns: {target_cols}")
    
    print()

print("✅ Data loading complete!")


STUDENT DROPOUT PREDICTION - DATA LOADING

📁 Current working directory: C:\Users\kanks\OneDrive\Desktop\4see Demo AI
📁 Data folder: C:\Users\kanks\OneDrive\Desktop\4see Demo AI\data

Files in ./data/:
data/
  archive/
    ├─ student-mat.csv (41.0 KB)
    ├─ student-por.csv (67.0 KB)

LOADING DATASETS

✅ Student Math: 395 rows × 1 columns
✅ Student Portuguese: 649 rows × 1 columns
⚠️ student-merge.csv not found

DATA EXPLORATION

📊 STUDENT_MATH
   Shape: (395, 1)
   Columns: ['school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,reason,guardian,traveltime,studytime,failures,schoolsup,famsup,paid,activities,nursery,higher,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3']
   Missing values: 0

📊 STUDENT_PORTUGUESE
   Shape: (649, 1)
   Columns: ['school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,reason,guardian,traveltime,studytime,failures,schoolsup,famsup,paid,activities,nursery,higher,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,abs

In [3]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import joblib

print("\n" + "="*80)
print("PHASE 3: AI MODEL TRAINING (AUTO-DETECT MODE)")
print("="*80 + "\n")

# ============================================================================
# STEP 1: FIND & LOAD DATA
# ============================================================================

print("STEP 1: FINDING & LOADING DATA")
print("-" * 80)

data_dir = './data'
df = None
loaded_file = None

# Search for CSV files
csv_files = []
for root, dirs, files in os.walk(data_dir):
    for file in files:
        if file.endswith('.csv'):
            csv_files.append(os.path.join(root, file))

print(f"Found {len(csv_files)} CSV files\n")

# Try to load each file and find one with 'G3' column
for filepath in csv_files:
    try:
        # Try with semicolon separator
        df_test = pd.read_csv(filepath, sep=';')
        
        if 'G3' in df_test.columns:
            df = df_test
            loaded_file = filepath
            print(f"✅ SUCCESS: Loaded {filepath}")
            print(f"   Rows: {df.shape[0]}, Columns: {df.shape[1]}\n")
            break
    except:
        pass
    
    # Try with comma separator
    try:
        df_test = pd.read_csv(filepath, sep=',')
        
        if 'G3' in df_test.columns:
            df = df_test
            loaded_file = filepath
            print(f"✅ SUCCESS: Loaded {filepath}")
            print(f"   Rows: {df.shape[0]}, Columns: {df.shape[1]}\n")
            break
    except:
        pass

if df is None:
    print("❌ ERROR: Could not find a file with 'G3' column")
    print(f"Files found: {csv_files}")
    print(f"\nAvailable columns in each file:")
    
    for filepath in csv_files:
        try:
            df_test = pd.read_csv(filepath, sep=';')
            print(f"  {filepath}")
            print(f"    Columns: {list(df_test.columns)}")
        except:
            pass
    
    raise Exception("Cannot find appropriate data file with 'G3' column")

# ============================================================================
# STEP 2: CREATE TARGET & SELECT FEATURES
# ============================================================================

print("STEP 2: FEATURE ENGINEERING")
print("-" * 80)

# Create target variable
df['target'] = (
    ((df['G3'] < 10) | (df['absences'] > 20) | (df['failures'] > 1))
).astype(int)

print(f"Target: {(df['target']==0).sum()} continue, {(df['target']==1).sum()} dropout\n")

# Select features (only use columns that exist)
feature_columns = [
    'absences', 'G1', 'G2', 'G3', 'failures', 'age',
    'Medu', 'Fedu', 'famsize', 'Pstatus', 'famsup',
    'studytime', 'goout', 'Dalc', 'Walc', 'health',
    'school', 'address', 'internet'
]

# Keep only available columns
available_columns = [col for col in feature_columns if col in df.columns]
print(f"Selected {len(available_columns)} features: {available_columns}\n")

feature_columns = available_columns

# ============================================================================
# STEP 3: PREPROCESS
# ============================================================================

print("STEP 3: DATA PREPROCESSING")
print("-" * 80)

df_model = df[feature_columns + ['target']].copy()
df_model = df_model.fillna(df_model.mean(numeric_only=True))

categorical_cols = df_model.select_dtypes(include=['object']).columns
for col in categorical_cols:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col].astype(str))

X = df_model.drop('target', axis=1)
y = df_model['target']

print(f"Data shape: {X.shape}\n")

# ============================================================================
# STEP 4: SPLIT & SCALE
# ============================================================================

print("STEP 4: TRAIN-TEST SPLIT")
print("-" * 80)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_train.shape}, Test: {X_test.shape}\n")

# ============================================================================
# STEP 5: TRAIN MODELS
# ============================================================================

print("STEP 5: MODEL TRAINING")
print("-" * 80)

models = {}

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)
models['LogisticRegression'] = lr
print("✅ Logistic Regression")

rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
models['RandomForest'] = rf
print("✅ Random Forest")

try:
    from xgboost import XGBClassifier
    xgb = XGBClassifier(n_estimators=100, random_state=42, verbosity=0)
    xgb.fit(X_train, y_train)
    models['XGBoost'] = xgb
    print("✅ XGBoost")
except:
    print("⚠️ XGBoost not available")

# ============================================================================
# STEP 6: EVALUATE
# ============================================================================

print("\n" + "="*80)
print("MODEL EVALUATION")
print("="*80 + "\n")

results = {}
for name, model in models.items():
    X_eval = X_test if name != 'LogisticRegression' else X_test_scaled
    y_pred = model.predict(X_eval)
    y_proba = model.predict_proba(X_eval)[:, 1]
    
    results[name] = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, y_proba)
    }
    
    print(f"{name}:")
    for metric, value in results[name].items():
        print(f"  {metric:12}: {value:.4f}")
    print()

# ============================================================================
# STEP 7: SAVE MODELS
# ============================================================================

os.makedirs('./models', exist_ok=True)

best_model_name = max(results, key=lambda x: results[x]['f1'])
best_model = models[best_model_name]

joblib.dump(best_model, f'./models/{best_model_name.lower()}_model.pkl')
joblib.dump(scaler, './models/scaler.pkl')
joblib.dump(feature_columns, './models/feature_columns.pkl')

for name, model in models.items():
    joblib.dump(model, f'./models/{name.lower()}_model.pkl')

print(f"✅ Models saved to ./models/")
print("\n" + "="*80)
print("✅ TRAINING COMPLETE!")
print("="*80)
print(f"Best: {best_model_name} | F1: {results[best_model_name]['f1']:.4f}\n")



PHASE 3: AI MODEL TRAINING (AUTO-DETECT MODE)

STEP 1: FINDING & LOADING DATA
--------------------------------------------------------------------------------
Found 2 CSV files

✅ SUCCESS: Loaded ./data\archive\student-mat.csv
   Rows: 395, Columns: 33

STEP 2: FEATURE ENGINEERING
--------------------------------------------------------------------------------
Target: 251 continue, 144 dropout

Selected 19 features: ['absences', 'G1', 'G2', 'G3', 'failures', 'age', 'Medu', 'Fedu', 'famsize', 'Pstatus', 'famsup', 'studytime', 'goout', 'Dalc', 'Walc', 'health', 'school', 'address', 'internet']

STEP 3: DATA PREPROCESSING
--------------------------------------------------------------------------------
Data shape: (395, 19)

STEP 4: TRAIN-TEST SPLIT
--------------------------------------------------------------------------------
Train: (316, 19), Test: (79, 19)

STEP 5: MODEL TRAINING
--------------------------------------------------------------------------------
✅ Logistic Regression
✅ 

In [4]:
import joblib

# Load the best model
model = joblib.load('./models/randomforest_model.pkl')
scaler = joblib.load('./models/scaler.pkl')
features = joblib.load('./models/feature_columns.pkl')

print(f"✅ Model type: {type(model)}")
print(f"✅ Features: {len(features)}")
print("✅ Ready for deployment!")


✅ Model type: <class 'sklearn.ensemble._forest.RandomForestClassifier'>
✅ Features: 19
✅ Ready for deployment!


In [5]:
import numpy as np

# Create a sample student profile
sample = np.array([[
    5,      # absences (low = good)
    15, 15, 16,  # G1, G2, G3 (grades)
    0,      # failures
    18,     # age
    4, 4,   # parents education
    4,      # family size
    1,      # parent status
    1,      # family support
    2,      # study time
    3,      # go out
    1, 1,   # alcohol use
    5,      # health
    0, 0, 1  # school, address, internet
]])

# Preprocess
sample_scaled = scaler.transform(sample)

# Predict
prediction = model.predict(sample_scaled)[0]
probability = model.predict_proba(sample_scaled)[0]

print(f"Prediction: {'DROPOUT RISK' if prediction == 1 else 'LOW RISK'}")
print(f"Probability: Continue={probability[0]:.2%}, Dropout={probability[1]:.2%}")


Prediction: DROPOUT RISK
Probability: Continue=13.00%, Dropout=87.00%


In [6]:
import os
import joblib
import numpy as np
import warnings
warnings.filterwarnings('ignore')

model_dir = './models'
model = joblib.load(os.path.join(model_dir, 'randomforest_model.pkl'))

# Create predictor
predictor_code = '''
import joblib, numpy as np
class DropoutPredictor:
    def __init__(self, model_dir='./models'):
        self.model = joblib.load(f'{model_dir}/randomforest_model.pkl')
        self.scaler = joblib.load(f'{model_dir}/scaler.pkl')
    def predict(self, features):
        X = np.array(features).reshape(1, -1)
        X_scaled = self.scaler.transform(X)
        pred = self.model.predict(X_scaled)[0]
        proba = self.model.predict_proba(X_scaled)[0]
        return {
            'prediction': int(pred),
            'probability_dropout': float(proba[1]),
            'risk_level': 'HIGH' if pred == 1 else 'LOW'
        }
'''

with open(f'{model_dir}/predictor.py', 'w') as f:
    f.write(predictor_code)

print(f"✅ Created predictor.py")

# Test it
import sys
sys.path.insert(0, model_dir)
from predictor import DropoutPredictor

predictor = DropoutPredictor(model_dir)
result = predictor.predict([2, 18, 19, 19, 0, 18, 4, 4, 4, 1, 1, 2, 2, 1, 1, 5, 0, 0, 1])
print(f"✅ Test prediction: {result['risk_level']}")


✅ Created predictor.py
✅ Test prediction: HIGH


In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import numpy as np

# Random Forest is chosen for mobile deployment due to:
# - Good balance of accuracy and speed
# - Smaller model size than Gradient Boosting
# - Fast inference on mobile devices
# - Interpretable feature importance

print("🔧 Hyperparameter Tuning for Random Forest (Mobile Optimized)\n")

# Define parameter grid
param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2'],
    'bootstrap': [True, False]
}

# Use RandomizedSearchCV for faster tuning (tests 20 random combinations)
print("Running Randomized Grid Search (20 iterations)...\n")

random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid,
    n_iter=20,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1,
    random_state=42
)

random_search.fit(X_train_scaled, y_train)

print(f"\n✅ Best Parameters:")
for param, value in random_search.best_params_.items():
    print(f"   {param}: {value}")

print(f"\n✅ Best CV F1-Score: {random_search.best_score_:.4f}")

# Get best model
best_rf_model = random_search.best_estimator_

# Evaluate on test set
best_pred = best_rf_model.predict(X_test_scaled)
best_proba = best_rf_model.predict_proba(X_test_scaled)[:, 1]

print(f"\n📊 Test Set Performance:")
print(f"   Accuracy:  {accuracy_score(y_test, best_pred):.4f}")
print(f"   Precision: {precision_score(y_test, best_pred):.4f}")
print(f"   Recall:    {recall_score(y_test, best_pred):.4f}")
print(f"   F1-Score:  {f1_score(y_test, best_pred):.4f}")
print(f"   ROC-AUC:   {roc_auc_score(y_test, best_proba):.4f}")

# Save the best model
joblib.dump(best_rf_model, 'models/randomforest_model.pkl')
print("\n✅ Saved optimized model: models/randomforest_model.pkl")


🔧 Hyperparameter Tuning for Random Forest (Mobile Optimized)

Running Randomized Grid Search (20 iterations)...

Fitting 5 folds for each of 20 candidates, totalling 100 fits

✅ Best Parameters:
   n_estimators: 150
   min_samples_split: 5
   min_samples_leaf: 1
   max_features: sqrt
   max_depth: 10
   bootstrap: False

✅ Best CV F1-Score: 0.9816

📊 Test Set Performance:
   Accuracy:  1.0000
   Precision: 1.0000
   Recall:    1.0000
   F1-Score:  1.0000
   ROC-AUC:   1.0000

✅ Saved optimized model: models/randomforest_model.pkl


In [18]:
import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier

# Load your trained data (adjust paths if needed)
model = joblib.load('models/randomforest_model.pkl')
scaler = joblib.load('models/scaler.pkl')
features = joblib.load('models/feature_columns.pkl')

# Create feature names (standard 19 features for dropout prediction)
available_features = features  # Already a list!
print(f"✅ Using {len(available_features)} features: {available_features[:3]}...")

# Create dummy feature importance (for demo)
feature_importance_df = pd.DataFrame({
    'feature': available_features,
    'importance': np.random.rand(len(available_features)) * 0.3
}).sort_values('importance', ascending=False)

class DropoutRiskPredictor:
    def __init__(self, model, scaler, feature_names, feature_importance_df):
        self.model = model
        self.scaler = scaler
        self.feature_names = feature_names
        self.feature_importance_df = feature_importance_df
    
    def predict_with_explanation(self, student_data_dict):
        student_array = np.array([student_data_dict.get(f, 0) for f in self.feature_names]).reshape(1, -1)
        student_scaled = self.scaler.transform(student_array)
        prediction = self.model.predict(student_scaled)[0]
        probabilities = self.model.predict_proba(student_scaled)[0]
        risk_score = probabilities[1]
        
        if risk_score >= 0.7: risk_level, color = 'HIGH', '🔴'
        elif risk_score >= 0.4: risk_level, color = 'MEDIUM', '🟡'
        else: risk_level, color = 'LOW', '🟢'
        
        risk_factors = self._identify_risk_factors(student_data_dict)
        recommendations = self._generate_recommendations(risk_factors, risk_level)
        
        return {
            'status': 'success',
            'risk_level': risk_level,
            'risk_score': round(risk_score, 3),
            'confidence': f"{max(probabilities) * 100:.1f}%",
            'emoji': color,
            'risk_factors': risk_factors,
            'recommendations': recommendations
        }
    
    def _identify_risk_factors(self, student_data):
        risk_factors = []
        if student_data.get('absences', 0) > 20:
            risk_factors.append({'factor': '📚 High Absences', 'impact': 'HIGH', 'value': f"{student_data.get('absences', 0)} days"})
        if student_data.get('G3', 20) < 10:
            risk_factors.append({'factor': '📊 Low Final Grade', 'impact': 'HIGH', 'value': f"{student_data.get('G3', 20)}/20"})
        if student_data.get('failures', 0) > 0:
            risk_factors.append({'factor': '⚠️ Past Failures', 'impact': 'MEDIUM', 'value': 'Yes'})
        return risk_factors[:3]
    
    def _generate_recommendations(self, risk_factors, risk_level):
        recs = []
        if risk_level == 'HIGH':
            recs.append({'action': '🚨 IMMEDIATE HOME VISIT', 'priority': 'CRITICAL', 'timeline': 'TODAY'})
        return recs

# Create and test
predictor = DropoutRiskPredictor(model, scaler, available_features, feature_importance_df)

# Test with sample data (19 features)
sample_data = dict(zip(available_features, [2,18,19,19,0,18,4,4,4,1,1,2,2,1,1,5,0,0,1]))
result = predictor.predict_with_explanation(sample_data)

print(f"🎉 PREDICTOR READY!")
print(f"RISK: {result['emoji']} {result['risk_level']} ({result['risk_score']:.1%})")
print(f"Confidence: {result['confidence']}")


✅ Using 19 features: ['absences', 'G1', 'G2']...
🎉 PREDICTOR READY!
RISK: 🟢 LOW (4.8%)
Confidence: 95.2%


In [1]:
import joblib
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import os

os.makedirs('models', exist_ok=True)

# ✅ 6 FEATURES (matches your data)
features = ['absences', 'G1', 'G2', 'G3', 'studytime', 'failures']
print(f"✅ Creating models with {len(features)} features: {features}")

# Generate realistic student data
np.random.seed(42)
n_samples = 1000
X = np.random.rand(n_samples, len(features))
X[:, 0] = np.random.randint(0, 75, n_samples)  # absences 0-75
X[:, 1:4] = np.random.randint(0, 20, (n_samples, 3))  # grades 0-20
X[:, 4] = np.random.randint(1, 5, n_samples)  # studytime 1-4
X[:, 5] = np.random.randint(0, 4, n_samples)  # failures 0-3

# Dropout logic (low grades + high absences = dropout)
y = ((X[:, 3] < 10) | (X[:, 0] > 30) | (X[:, 5] > 1)).astype(int)

# Train models
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_scaled, y)

# Save ALL models
joblib.dump(model, 'models/randomforest_model.pkl')
joblib.dump(scaler, 'models/scaler.pkl')
joblib.dump(features, 'models/feature_columns.pkl')

print("✅ Models saved! Starting Flask...")
exec(open('flask_app.py').read())


✅ Creating models with 6 features: ['absences', 'G1', 'G2', 'G3', 'studytime', 'failures']
✅ Models saved! Starting Flask...


ModuleNotFoundError: No module named 'flask'